# 01 – Preprocessing Tokopedia (Individual)

Notebook ini digunakan setiap anggota kelompok untuk:
- Membaca file CSV hasil scraping per orang (per kategori & brand).
- Menambahkan kolom `category`, `brand`, dan `condition`.
- Membersihkan kolom `price`, `rating`, `review_count`, dan `sold`.
- Menurunkan kolom `sold_text` dan `sold_num`.
- Menyimpan output bersih per orang ke file, misalnya `laptop-olive.csv`.

Notebook ini belum menggabungkan data seluruh anggota kelompok. Penggabungan dilakukan di notebook `02_preprocessing_tokopedia_merge_all.ipynb`.

In [1]:
# bismillah lancar tanpa eror

import pandas as pd
import re
from pathlib import Path

base_path = Path("../data/raw")
files = [
    "laptop_advan_baru.csv",
    "laptop_advan_bekas.csv",
    "laptop_apple_baru.csv",
    "laptop_apple_bekas.csv",
    "laptop_asus_baru.csv",
    "laptop_asus_bekas.csv",
    "laptop_axioo_baru.csv",
    "laptop_axioo_bekas.csv",
    "laptop_dell_baru.csv",
    "laptop_dell_bekas.csv",
    "laptop_lenovo_baru.csv",
    "laptop_lenovo_bekas.csv",
    "laptop_msi_baru.csv",
    "laptop_msi_bekas.csv",
    "laptop_zyrex_baru.csv",
    "laptop_zyrex_bekas.csv",
]

dfs = []
for fname in files:
    f = base_path / fname
    nama = f.stem                      
    parts = nama.split("_")
    category = parts[0]
    brand = parts[1]
    condition = parts[2]

    df_tmp = pd.read_csv(f)
    df_tmp['category'] = category.capitalize()
    df_tmp['brand'] = brand.capitalize()
    df_tmp['condition'] = condition.capitalize()
    dfs.append(df_tmp)

# gabungno semuane
df = pd.concat(dfs, ignore_index=True)

# cek awal
df

,web_scraper_order,web_scraper_start_url,product_link,product_name,price,rating,review_count,sold,seller,category,brand,condition
0,1774591472-1,https://www.tokopedia.com/search?condition=1&n...,https://www.tokopedia.com/agrescilegon/advan-t...,Advan Tbook Intel N100 RAM 8GB SSD 128GB 14.0'...,Rp3.799.000,NaN,NaN,NaN,NaN,Laptop,Advan,Baru
1,1774591477-2,https://www.tokopedia.com/search?condition=1&n...,https://www.tokopedia.com/best-buyit/advan-sou...,NaN,NaN,NaN,NaN,NaN,NaN,Laptop,Advan,Baru
2,1774591480-3,https://www.tokopedia.com/search?condition=1&n...,https://www.tokopedia.com/pandacom/laptop-adva...,LAPTOP ADVAN SOULMATE PLUS 1405-L24D-R53S1 AMD...,Rp4.305.100,5.0,2 rating • 1 ulasan,Terjual 3,Pandacom_NEW,Laptop,Advan,Baru
3,1774591484-4,https://www.tokopedia.com/search?condition=1&n...,https://www.tokopedia.com/fijanstore/advan-net...,Advan netbook (atau lebih tepatnya laptop entr...,Rp2.000.000,NaN,NaN,NaN,fijanstore,Laptop,Advan,Baru
4,1774591488-5,https://www.tokopedia.com/search?condition=1&n...,https://www.tokopedia.com/jedata/advan-t-book-...,Advan T-Book Transformers Intel Core N100 4GB/...,Rp3.394.000,NaN,NaN,NaN,JET COMPUTER,Laptop,Advan,Baru
...,...,...,...,...,...,...,...,...,...,...,...,...
2872,1774627262-176,https://www.tokopedia.com/search?condition=2&n...,https://www.tokopedia.com/drcomkomputer/laptop...,"Laptop Zyrex Sky Cruiser 20, i3-10110, 10Th, R...",Rp2.850.000,NaN,NaN,1 barang berhasil terjual,DrCom_NEW,Laptop,Zyrex,Bekas
2873,1774627265-177,https://www.tokopedia.com/search?condition=2&n...,https://www.tokopedia.com/drcomkomputer/zyrex-...,"Zyrex ASXU, Amd A9-9400, Gen 7Th, Ram 4/128Gb,...",Rp1.850.000,5.0,3 rating • 2 ulasan,Terjual 3,DrCom_NEW,Laptop,Zyrex,Bekas
2874,1774627269-178,https://www.tokopedia.com/search?condition=2&n...,https://www.tokopedia.com/drcomkomputer/notebo...,Notebook Zyrex Sky 232/ Celeron - N3350/ HD Gr...,Rp1.550.000,5.0,12 rating • 7 ulasan,Terjual 22,DrCom_NEW,Laptop,Zyrex,Bekas
2875,1774627272-179,https://www.tokopedia.com/search?condition=2&n...,https://www.tokopedia.com/r2nstore/laptop-zyre...,Laptop Zyrex Bunaken Ex Kantor,Rp1.700.000,NaN,NaN,NaN,JG COMP_NEW,Laptop,Zyrex,Bekas


In [2]:
# ngisi kolom sng kosong
df['rating'] = df['rating'].fillna('0')
df['review_count'] = df['review_count'].fillna('0 rating • 0 ulasan')
df['sold'] = df['sold'].fillna('Terjual 0')

# mastino 'sold' wes string
df['sold'] = df['sold'].astype(str)

# gae kolom 'sold' versi rapi (misal "1 barang berhasil terjual" -> "Terjual 1")
df['sold_text'] = df['sold']

# ngeubah sng awale "30+ barang berhasil terjual" / "1 barang berhasil terjual"
# dadi "Terjual 30+" / "Terjual 1"
df['sold_text'] = df['sold_text'].str.replace(
    r'(\d+\+?)\s*barang berhasil terjual',
    r'Terjual \1',
    regex=True
)

# fungsi gae ngitung sold_num, termasuk kasus "3 rb+"
def parse_sold_num(s):
    if not isinstance(s, str):
        return 0
    s = s.lower().strip()

    # kasus "3 rb+" atau "3 rb"
    m = re.search(r'(\d+)\s*rb', s)
    if m:
        return int(m.group(1)) * 1000  # 3 rb didadino 3000

    # kasus umum: ambil angka pertama ("Terjual 30+", "Terjual 17", "Terjual 1")
    m = re.search(r'(\d+)', s)
    if m:
        return int(m.group(1))

    return 0

# gae kolom 'sold' versi tanpa + (misal Terjual 60+) ben enak ngitung e nnti
df['sold_num'] = df['sold_text'].apply(parse_sold_num)

# normalisasi 'price' nak numerik
df['price'] = (
    df['price'].astype(str)
    .str.replace('Rp', '', regex=False)
    .str.replace(' ', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# drop baris sng gaonok seller atau price
df = df.dropna(subset=['seller', 'price'])

# alhamdulillah selesai 🙏✨
output_dir = Path("../data/processed/individual")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "laptop-olive.csv"
df.to_csv(output_path, index=False)

# cek akhir
df

,web_scraper_order,web_scraper_start_url,product_link,product_name,price,rating,review_count,sold,seller,category,brand,condition,sold_text,sold_num
2,1774591480-3,https://www.tokopedia.com/search?condition=1&n...,https://www.tokopedia.com/pandacom/laptop-adva...,LAPTOP ADVAN SOULMATE PLUS 1405-L24D-R53S1 AMD...,4305100.0,5.0,2 rating • 1 ulasan,Terjual 3,Pandacom_NEW,Laptop,Advan,Baru,Terjual 3,3
3,1774591484-4,https://www.tokopedia.com/search?condition=1&n...,https://www.tokopedia.com/fijanstore/advan-net...,Advan netbook (atau lebih tepatnya laptop entr...,2000000.0,0,0 rating • 0 ulasan,Terjual 0,fijanstore,Laptop,Advan,Baru,Terjual 0,0
4,1774591488-5,https://www.tokopedia.com/search?condition=1&n...,https://www.tokopedia.com/jedata/advan-t-book-...,Advan T-Book Transformers Intel Core N100 4GB/...,3394000.0,0,0 rating • 0 ulasan,Terjual 0,JET COMPUTER,Laptop,Advan,Baru,Terjual 0,0
5,1774591492-6,https://www.tokopedia.com/search?condition=1&n...,https://www.tokopedia.com/greencomputerbalikpa...,Laptop Advan Soulmate Intel N4020 ram 4GB MMC ...,3090000.0,0,0 rating • 0 ulasan,Terjual 0,greencomputerbpn,Laptop,Advan,Baru,Terjual 0,0
6,1774591496-7,https://www.tokopedia.com/search?condition=1&n...,https://www.tokopedia.com/gigacom-/advan-soulm...,ADVAN SOULMATE T 1405 L24I NSNS1 UPGRADE VERIS...,2339000.0,0,0 rating • 0 ulasan,1 barang berhasil terjual,GIGACOM TANGCITY,Laptop,Advan,Baru,Terjual 1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2872,1774627262-176,https://www.tokopedia.com/search?condition=2&n...,https://www.tokopedia.com/drcomkomputer/laptop...,"Laptop Zyrex Sky Cruiser 20, i3-10110, 10Th, R...",2850000.0,0,0 rating • 0 ulasan,1 barang berhasil terjual,DrCom_NEW,Laptop,Zyrex,Bekas,Terjual 1,1
2873,1774627265-177,https://www.tokopedia.com/search?condition=2&n...,https://www.tokopedia.com/drcomkomputer/zyrex-...,"Zyrex ASXU, Amd A9-9400, Gen 7Th, Ram 4/128Gb,...",1850000.0,5.0,3 rating • 2 ulasan,Terjual 3,DrCom_NEW,Laptop,Zyrex,Bekas,Terjual 3,3
2874,1774627269-178,https://www.tokopedia.com/search?condition=2&n...,https://www.tokopedia.com/drcomkomputer/notebo...,Notebook Zyrex Sky 232/ Celeron - N3350/ HD Gr...,1550000.0,5.0,12 rating • 7 ulasan,Terjual 22,DrCom_NEW,Laptop,Zyrex,Bekas,Terjual 22,22
2875,1774627272-179,https://www.tokopedia.com/search?condition=2&n...,https://www.tokopedia.com/r2nstore/laptop-zyre...,Laptop Zyrex Bunaken Ex Kantor,1700000.0,0,0 rating • 0 ulasan,Terjual 0,JG COMP_NEW,Laptop,Zyrex,Bekas,Terjual 0,0
